<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 31 · Market-Based Valuation

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

import datetime as dt
import math
from dataclasses import dataclass
from typing import Any, Protocol

import numpy as np
import pandas as pd

from dxlib import *


class Pricer(Protocol):
    def value(self, spot: float) -> tuple[float, float]: ...


snapshot_path = PROJECT_ROOT / 'data' / 'spx_options_snapshot.csv'
snap = load_spx_snapshot(snapshot_path) if snapshot_path.exists() else None
if snap is not None:
    short_expiry = dt.date(2026, 3, 20)
    long_expiries = [dt.date(2026, 6, 18), dt.date(2026, 12, 18)]
    short_surface_df = select_small_surface(snap, [short_expiry], rate=0.03)
    long_surface_df = select_small_surface(snap, long_expiries, rate=0.03)
    surface_df = long_surface_df
    paths = 15_000
    steps_per_year = 320
    rate = 0.03
    h_params = HestonParams(
        kappa=2.6,
        theta=0.046,
        vol_of_vol=0.9,
        rho=-0.67,
        v0=0.018,
    )
    params = h_params
    local_map = {
        dt.date(2026, 6, 18): (0.0606, 0.0128, -0.732),
        dt.date(2026, 12, 18): (0.0484, 0.0149, -0.763),
    }
    seed = 11


## Why “Market-Based” Matters

Execute the code examples below.


## The Dataset: A Snapshot of Index Options Quotes

Execute the code examples below.


In [ ]:
import pandas as pd

raw = pd.read_csv("../data/spx_options_snapshot.csv")

raw.shape

In [ ]:
def load_spx_snapshot(path):
    df = pd.read_csv(path)
    df["PUTCALLIND"] = df["PUTCALLIND"].astype(str).str.strip().str.upper()
    df["EXPIR_DATE"] = pd.to_datetime(df["EXPIR_DATE"]).dt.date
    df["HSTCLSDATE"] = pd.to_datetime(df["HSTCLSDATE"]).dt.date
    df["BID"] = df["BID"].astype(float)
    df["ASK"] = df["ASK"].astype(float)
    df["MID"] = 0.5 * (df["BID"] + df["ASK"])
    df["SPREAD"] = df["ASK"] - df["BID"]
    df["HALF_SPREAD"] = 0.5 * df["SPREAD"]
    pricing_date = df["HSTCLSDATE"].iloc[0]
    return OptionsSnapshot(pricing_date=pricing_date, raw=df)

## Put-Call Parity and Forward Levels

Execute the code examples below.


## Implied Volatility via Black-76

Execute the code examples below.


## Selecting a Small Surface (3 Expiries × 5 Strikes)

Execute the code examples below.


## Figures: Smiles and Term Structure

Execute the code examples below.


## A Small Implied-Vol Surface Object

Execute the code examples below.


## Horizon-Split Calibration: Jump Diffusion then Heston

Execute the code examples below.


## Short Horizon: Jump Diffusion on One Expiry

Execute the code examples below.


In [ ]:
jd_params, jd_fit = calibrate_jump_diffusion_single_expiry(
    short_surface_df,  # one expiry, five strikes
    rate=0.03,
    paths=15_000,
    steps_per_year=320,
    n_candidates=40,
    n_refine=25,
    seed=11,
)

## Longer Horizon: Heston Global Step (Self-Contained)

Execute the code examples below.


In [ ]:
params, fit = calibrate_heston_global(
    surface_df,
    rate=0.03,
    paths=15_000,
    steps_per_year=320,
    n_candidates=40,
    n_refine=25,
    seed=11,
)

## Local Step: Per-Expiry Refinement

Execute the code examples below.


## Calibration Fit Figure (Longer Horizons)

Execute the code examples below.


## Pricing with the Calibrated Heston Model (Monte Carlo)

Execute the code examples below.


## Interactive Session: Short-Horizon JD, Long-Horizon Heston

Execute the code examples below.


In [ ]:
import datetime as dt

import numpy as np

from dxlib import (
    AmericanPut,
    HestonParams,
    FlatDiscounting,
    build_time_grid,
    calibrate_heston_global,
    calibrate_heston_local_theta_v0_rho,
    calibrate_jump_diffusion_single_expiry,
    evaluate_heston_fit_table,
    load_spx_snapshot,
    lsm_american_put_from_paths,
    mc_call_prices,
    select_small_surface,
    simulate_heston_spot_paths,
    spot_from_forwards,
)

In [ ]:
snap = load_spx_snapshot("../data/spx_options_snapshot.csv")

short_expiry = dt.date(2026, 3, 20)

long_expiries = [
    dt.date(2026, 6, 18),
    dt.date(2026, 12, 18),
]

short_surface_df = select_small_surface(
    snap,
    [short_expiry],
    rate=0.03,
    calibrate_to="CALL",
    moneyness_targets=[0.8, 0.9, 1.0, 1.05, 1.1],
)

long_surface_df = select_small_surface(
    snap,
    long_expiries,
    rate=0.03,
    calibrate_to="CALL",
    moneyness_targets=[0.8, 0.9, 1.0, 1.05, 1.1],
)

round(spot_from_forwards(short_surface_df), 2), round(
    spot_from_forwards(long_surface_df), 2
)

In [ ]:
jd_params, jd_fit = calibrate_jump_diffusion_single_expiry(
    short_surface_df,
    rate=0.03,
    paths=15_000,
    steps_per_year=320,
    n_candidates=40,
    n_refine=25,
    seed=11,
)

tuple(
    round(float(x), 5)
    for x in (
        jd_params.volatility,
        jd_params.jump_intensity,
        jd_params.jump_mean,
        jd_params.jump_std,
    )
)

In [ ]:
h_params, fit_global = calibrate_heston_global(
    long_surface_df,
    rate=0.03,
    paths=15_000,
    steps_per_year=320,
    n_candidates=40,
    n_refine=25,
    seed=11,
)

tuple(
    round(float(x), 5)
    for x in (
        h_params.kappa,
        h_params.theta,
        h_params.vol_of_vol,
        h_params.rho,
        h_params.v0,
    )
)

In [ ]:
expiry = dt.date(2026, 6, 18)

mask = long_surface_df["EXPIR_DATE"] == expiry

sub = long_surface_df[mask].sort_values("STRIKE")

ttm = float(sub["TTM"].iloc[0])

strikes = sub["STRIKE"].to_numpy(dtype=float)

df = float(sub["DF"].iloc[0])

theta, v0, rho = local_map[expiry]

local_params = HestonParams(
    kappa=h_params.kappa,
    theta=theta,
    vol_of_vol=h_params.vol_of_vol,
    rho=rho,
    v0=v0,
)

spot0 = spot_from_forwards(long_surface_df)

spot_paths, _ = simulate_heston_spot_paths(
    spot=spot0,
    rate=0.03,
    maturity=ttm,
    steps=100,
    paths=50_000,
    params=local_params,
    seed=7,
)

model_calls = mc_call_prices(
    spot_paths,
    strikes,
    discount_factor=df,
    forward=float(sub["FWD"].iloc[0]),
)

mkt_calls = sub["CALL_MID"].to_numpy(dtype=float)

table = np.column_stack((strikes, mkt_calls, model_calls))

np.round(table, 2)

In [ ]:
grid = build_time_grid(maturity=ttm, steps=100)

disc = FlatDiscounting(rate=0.03)

am = AmericanPut(strike=float(strikes[2]))

res = lsm_american_put_from_paths(
    spot_paths,
    strike=am.strike,
    discounting=disc,
    time_grid=grid,
    basis_degree=2,
)

round(float(res["price"]), 3), round(float(res["stderr"]), 4)

## Where We Are Heading Next

Execute the code examples below.


## Appendix: `dxlib` Market-Based Valuation Source Code

Execute the code examples below.


## `code/dxlib/black_scholes.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 31 - Market-Based Valuation.

Black-Scholes (Black-76 forward form) pricing and implied volatility.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

import math

__all__ = [
    "norm_cdf",
    "norm_pdf",
    "bs_price_forward",
    "implied_vol_forward",
]


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(float(x) / math.sqrt(2.0)))


def norm_pdf(x: float) -> float:
    x = float(x)
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_price_forward(
    forward: float,
    strike: float,
    maturity: float,
    volatility: float,
    *,
    option_type: str = "call",
    discount_factor: float = 1.0,
) -> float:
    """
    Black-76 price for an option on a forward with deterministic discounting.
    """

    fwd = float(forward)
    k = float(strike)
    t = float(maturity)
    vol = float(volatility)
    df = float(discount_factor)
    opt = str(option_type).strip().lower()

    if fwd <= 0 or k <= 0:
        raise ValueError("forward and strike must be positive")
    if t < 0:
        raise ValueError("maturity must be non-negative")
    if vol < 0:
        raise ValueError("volatility must be non-negative")
    if df <= 0:
        raise ValueError("discount_factor must be positive")
    if opt not in {"call", "put"}:
        raise ValueError("option_type must be 'call' or 'put'")

    if t == 0.0 or vol == 0.0:
        intrinsic = max(fwd - k, 0.0)
        if opt == "put":
            intrinsic = max(k - fwd, 0.0)
        return df * intrinsic

    sqrt_t = math.sqrt(t)
    sig_sqrt = vol * sqrt_t
    d1 = (math.log(fwd / k) + 0.5 * vol * vol * t) / sig_sqrt
    d2 = d1 - sig_sqrt

    if opt == "call":
        value = df * (fwd * norm_cdf(d1) - k * norm_cdf(d2))
    else:
        value = df * (k * norm_cdf(-d2) - fwd * norm_cdf(-d1))

    return float(value)


def implied_vol_forward(
    price: float,
    forward: float,
    strike: float,
    maturity: float,
    *,
    option_type: str = "call",
    discount_factor: float = 1.0,
    tol: float = 1.0e-8,
    max_iter: int = 100,
) -> float:
    """
    Implied volatility for Black-76 with deterministic discounting.

    Uses a robust bisection method with an adaptive upper bracket.
    """

    target = float(price)
    if target < 0:
        raise ValueError("price must be non-negative")
    if maturity <= 0:
        raise ValueError("maturity must be positive")

    opt = str(option_type).strip().lower()
    df = float(discount_factor)
    intrinsic = bs_price_forward(
        forward,
        strike,
        maturity,
        0.0,
        option_type=opt,
        discount_factor=df,
    )
    if target < intrinsic:
        return 0.0

    low = 1.0e-8
    high = 1.0
    for _ in range(50):
        high_price = bs_price_forward(
            forward,
            strike,
            maturity,
            high,
            option_type=opt,
            discount_factor=df,
        )
        if high_price >= target:
            break
        high *= 2.0
    else:
        raise RuntimeError("Failed to bracket implied volatility")

    for _ in range(int(max_iter)):
        mid = 0.5 * (low + high)
        mid_price = bs_price_forward(
            forward,
            strike,
            maturity,
            mid,
            option_type=opt,
            discount_factor=df,
        )
        err = mid_price - target
        if abs(err) <= tol:
            return float(mid)
        if err > 0.0:
            high = mid
        else:
            low = mid

    return float(0.5 * (low + high))

## `code/dxlib/marketdata.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 31 - Market-Based Valuation.

Utilities to load an options snapshot and extract a small implied-vol surface.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import datetime as dt
import math
import sys
from dataclasses import dataclass
from pathlib import Path

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

import numpy as np
import pandas as pd

from .black_scholes import implied_vol_forward
from .time import time_to_maturity

__all__ = [
    "OptionsSnapshot",
    "load_spx_snapshot",
    "parity_table",
    "estimate_forward",
    "select_small_surface",
]


@dataclass(frozen=True, slots=True)
class OptionsSnapshot:
    """
    Cleaned options snapshot with metadata used throughout Chapter 31.
    """

    pricing_date: dt.date
    raw: pd.DataFrame


def load_spx_snapshot(path: str | Path) -> OptionsSnapshot:
    """
    Load and clean the SPX options snapshot CSV.
    """

    df = pd.read_csv(path)
    df = df.copy()
    df["PUTCALLIND"] = df["PUTCALLIND"].astype(str).str.strip().str.upper()
    df["EXPIR_DATE"] = pd.to_datetime(df["EXPIR_DATE"]).dt.date
    df["HSTCLSDATE"] = pd.to_datetime(df["HSTCLSDATE"]).dt.date
    df["BID"] = df["BID"].astype(float)
    df["ASK"] = df["ASK"].astype(float)
    df["MID"] = 0.5 * (df["BID"].astype(float) + df["ASK"].astype(float))
    df["SPREAD"] = df["ASK"] - df["BID"]
    df["HALF_SPREAD"] = 0.5 * df["SPREAD"]
    pricing_date = df["HSTCLSDATE"].iloc[0]
    return OptionsSnapshot(pricing_date=pricing_date, raw=df)


def parity_table(snapshot: OptionsSnapshot, expiry: dt.date) -> pd.DataFrame:
    """
    Merge calls and puts by strike for a single expiry.
    """

    df = snapshot.raw
    sub = df[df["EXPIR_DATE"] == expiry].copy()
    calls = sub[sub["PUTCALLIND"] == "CALL"].copy()
    puts = sub[sub["PUTCALLIND"] == "PUT"].copy()

    calls = calls.rename(
        columns={
            "MID": "CALL_MID",
            "BID": "CALL_BID",
            "ASK": "CALL_ASK",
            "SPREAD": "CALL_SPREAD",
            "HALF_SPREAD": "CALL_HALF_SPREAD",
            "DELTA": "CALL_DELTA",
            "MID_IV": "CALL_MID_IV",
        }
    )
    puts = puts.rename(
        columns={
            "MID": "PUT_MID",
            "BID": "PUT_BID",
            "ASK": "PUT_ASK",
            "SPREAD": "PUT_SPREAD",
            "HALF_SPREAD": "PUT_HALF_SPREAD",
            "DELTA": "PUT_DELTA",
            "MID_IV": "PUT_MID_IV",
        }
    )

    out = pd.merge(
        calls[
            [
                "EXPIR_DATE",
                "STRIKE_PRC",
                "CALL_BID",
                "CALL_ASK",
                "CALL_MID",
                "CALL_SPREAD",
                "CALL_HALF_SPREAD",
                "CALL_DELTA",
                "CALL_MID_IV",
            ]
        ],
        puts[
            [
                "EXPIR_DATE",
                "STRIKE_PRC",
                "PUT_BID",
                "PUT_ASK",
                "PUT_MID",
                "PUT_SPREAD",
                "PUT_HALF_SPREAD",
                "PUT_DELTA",
                "PUT_MID_IV",
            ]
        ],
        on=["EXPIR_DATE", "STRIKE_PRC"],
        how="inner",
    )
    out = out.rename(columns={"STRIKE_PRC": "STRIKE"})
    return out.sort_values("STRIKE").reset_index(drop=True)


def estimate_forward(
    parity: pd.DataFrame,
    *,
    discount_factor: float,
    min_price: float = 0.5,
    delta_band: tuple[float, float] = (0.2, 0.8),
) -> float:
    """
    Estimate the forward from put-call parity across a filtered strike band.
    """

    df = float(discount_factor)
    if df <= 0:
        raise ValueError("discount_factor must be positive")

    lo, hi = float(delta_band[0]), float(delta_band[1])
    work = parity.copy()
    work = work[(work["CALL_MID"] > min_price) & (work["PUT_MID"] > min_price)]
    work = work[(work["CALL_DELTA"] >= lo) & (work["CALL_DELTA"] <= hi)]
    if work.empty:
        raise ValueError("No suitable strikes to estimate the forward")

    fwd = work["STRIKE"] + (work["CALL_MID"] - work["PUT_MID"]) / df
    fwd = fwd.to_numpy(dtype=float)
    return float(np.median(fwd))


def select_small_surface(
    snapshot: OptionsSnapshot,
    expiries: list[dt.date],
    *,
    rate: float,
    moneyness_targets: list[float] | None = None,
    calibrate_to: str = "CALL",
    min_mid_price: float = 0.5,
    short_ttm_threshold: float | None = None,
    short_min_mid_price: float | None = None,
) -> pd.DataFrame:
    """
    Select a 3x5 quotes surface (calls + puts) for the book examples.
    """

    if moneyness_targets is None:
        moneyness_targets = [0.8, 0.9, 1.0, 1.1, 1.2]
    if min_mid_price <= 0:
        raise ValueError("min_mid_price must be positive")
    if short_ttm_threshold is not None and float(short_ttm_threshold) <= 0.0:
        raise ValueError("short_ttm_threshold must be positive")
    if short_min_mid_price is not None and float(short_min_mid_price) <= 0.0:
        raise ValueError("short_min_mid_price must be positive")

    def select_strikes(
        frame: pd.DataFrame,
        *,
        targets: list[float],
        mid_col: str,
        min_mid: float,
        n: int = 5,
    ) -> list[float]:
        eligible = frame[frame[mid_col] >= float(min_mid)].copy()
        if eligible.empty:
            raise ValueError("No eligible quotes after min_mid_price filter")

        strikes_out: list[float] = []
        for target in targets:
            work = eligible.copy()
            work["dist"] = (work["MNY"] - float(target)).abs()
            work = work.sort_values("dist")
            for strike in work["STRIKE"].to_numpy(dtype=float):
                if float(strike) not in strikes_out:
                    strikes_out.append(float(strike))
                    break
            if len(strikes_out) >= n:
                break

        if len(strikes_out) < n:
            remaining = eligible.copy()
            remaining = remaining[~remaining["STRIKE"].isin(strikes_out)]
            remaining = remaining.assign(
                abs_mny=(remaining["MNY"] - 1.0).abs(),
            )
            remaining = remaining.sort_values("abs_mny")
            for strike in remaining["STRIKE"].to_numpy(dtype=float):
                strikes_out.append(float(strike))
                if len(strikes_out) >= n:
                    break

        return sorted(strikes_out)[:n]

    results: list[pd.DataFrame] = []
    for expiry in expiries:
        parity = parity_table(snapshot, expiry)
        ttm = time_to_maturity(snapshot.pricing_date, expiry)
        expiry_min_mid = float(min_mid_price)
        if (
            short_ttm_threshold is not None
            and short_min_mid_price is not None
            and float(ttm) <= float(short_ttm_threshold)
        ):
            expiry_min_mid = max(expiry_min_mid, float(short_min_mid_price))
        df = math.exp(-float(rate) * float(ttm))
        fwd = estimate_forward(
            parity,
            discount_factor=df,
            min_price=expiry_min_mid,
        )

        parity = parity.copy()
        parity["TTM"] = float(ttm)
        parity["DF"] = float(df)
        parity["FWD"] = float(fwd)
        parity["MNY"] = parity["STRIKE"] / float(fwd)

        opt = str(calibrate_to).strip().upper()
        if opt not in {"CALL", "PUT"}:
            raise ValueError("calibrate_to must be 'CALL' or 'PUT'")

        base_targets = list(moneyness_targets)
        eligible = parity[parity[f"{opt}_MID"] >= float(expiry_min_mid)]
        max_mny = float(eligible["MNY"].max())
        if max_mny < 1.15:
            base_targets = [0.8, 0.9, 1.0, 1.05, 1.1]

        strikes = select_strikes(
            parity,
            targets=base_targets,
            mid_col=f"{opt}_MID",
            min_mid=expiry_min_mid,
            n=5,
        )
        small = parity[parity["STRIKE"].isin(strikes)].copy()

        ivs: list[float] = []
        for _, row in small.iterrows():
            price = float(row[f"{opt}_MID"])
            iv = implied_vol_forward(
                price=price,
                forward=float(row["FWD"]),
                strike=float(row["STRIKE"]),
                maturity=float(row["TTM"]),
                option_type="call" if opt == "CALL" else "put",
                discount_factor=float(row["DF"]),
            )
            ivs.append(iv)
        small["CALIBRATE_TO"] = opt
        small["IMPL_VOL"] = np.array(ivs, dtype=float)
        results.append(small)

    out = pd.concat(results, axis=0, ignore_index=True)
    return out

## `code/dxlib/volsurface.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 31 - Market-Based Valuation.

Simple implied-volatility surface built from a small set of smiles.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray

__all__ = ["ImpliedVolSurface"]

FloatArray = NDArray[np.float64]


@dataclass(frozen=True, slots=True)
class ImpliedVolSurface:
    """
    Bilinear interpolation on (time-to-maturity, moneyness) nodes.
    """

    maturities: FloatArray
    moneyness_grid: dict[float, FloatArray]
    iv_grid: dict[float, FloatArray]
    extrapolate: str = "flat"

    def __post_init__(self) -> None:
        mode = str(self.extrapolate).strip().lower()
        if mode not in {"flat", "error"}:
            raise ValueError("extrapolate must be 'flat' or 'error'")
        if self.maturities.ndim != 1 or self.maturities.size == 0:
            raise ValueError(
                "maturities must be a non-empty one-dimensional array"
            )
        if np.any(np.diff(self.maturities) <= 0):
            raise ValueError("maturities must be strictly increasing")
        for maturity in self.maturities:
            t = float(maturity)
            mny = self.moneyness_grid[t]
            iv = self.iv_grid[t]
            if mny.ndim != 1 or iv.ndim != 1 or mny.size != iv.size:
                raise ValueError("moneyness and iv grids must be aligned")
            if mny.size == 0 or np.any(np.diff(mny) <= 0):
                raise ValueError("moneyness nodes must be strictly increasing")

    @classmethod
    def from_frame(
        cls,
        frame,
        *,
        maturity_col: str = "TTM",
        moneyness_col: str = "MNY",
        iv_col: str = "IMPL_VOL",
        extrapolate: str = "flat",
    ) -> "ImpliedVolSurface":
        maturities = np.sort(frame[maturity_col].unique()).astype(float)
        mny_grid: dict[float, FloatArray] = {}
        iv_grid: dict[float, FloatArray] = {}
        for t in maturities:
            sub = frame[frame[maturity_col] == t].sort_values(moneyness_col)
            mny = sub[moneyness_col].to_numpy(dtype=float)
            iv = sub[iv_col].to_numpy(dtype=float)
            mny_grid[float(t)] = mny
            iv_grid[float(t)] = iv
        return cls(
            maturities=maturities,
            moneyness_grid=mny_grid,
            iv_grid=iv_grid,
            extrapolate=extrapolate,
        )

    def _interp_smile(self, maturity: float, moneyness: float) -> float:
        mode = str(self.extrapolate).strip().lower()
        mny = self.moneyness_grid[maturity]
        iv = self.iv_grid[maturity]
        if mode == "error":
            if moneyness < float(mny[0]) or moneyness > float(mny[-1]):
                raise ValueError("moneyness is outside surface node range")
        return float(np.interp(moneyness, mny, iv))

    def vol(self, maturity: float, moneyness: float) -> float:
        t = float(maturity)
        x = float(moneyness)
        if t <= 0:
            raise ValueError("maturity must be positive")
        if x <= 0:
            raise ValueError("moneyness must be positive")

        mode = str(self.extrapolate).strip().lower()
        t_grid = self.maturities
        if t <= float(t_grid[0]):
            t0 = float(t_grid[0])
            if mode == "error" and t < t0:
                raise ValueError("maturity is outside surface node range")
            return self._interp_smile(t0, x)
        if t >= float(t_grid[-1]):
            t1 = float(t_grid[-1])
            if mode == "error" and t > t1:
                raise ValueError("maturity is outside surface node range")
            return self._interp_smile(t1, x)

        idx = int(np.searchsorted(t_grid, t))
        t_lo = float(t_grid[idx - 1])
        t_hi = float(t_grid[idx])
        iv_lo = self._interp_smile(t_lo, x)
        iv_hi = self._interp_smile(t_hi, x)
        weight = (t - t_lo) / (t_hi - t_lo)
        return float((1.0 - weight) * iv_lo + weight * iv_hi)

## `code/dxlib/heston.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 31 - Market-Based Valuation.

Heston model helpers for Monte Carlo pricing and calibration.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import math
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from numpy.typing import NDArray

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

from .processes import HestonModel, build_time_grid

__all__ = ["HestonParams", "simulate_heston_spot_paths", "mc_call_prices"]

FloatArray = NDArray[np.float64]


@dataclass(frozen=True, slots=True)
class HestonParams:
    kappa: float
    theta: float
    vol_of_vol: float
    rho: float
    v0: float

    def __post_init__(self) -> None:
        if self.kappa <= 0:
            raise ValueError("kappa must be positive")
        if self.theta <= 0:
            raise ValueError("theta must be positive")
        if self.vol_of_vol < 0:
            raise ValueError("vol_of_vol must be non-negative")
        if not -1.0 <= self.rho <= 1.0:
            raise ValueError("rho must be in [-1, 1]")
        if self.v0 <= 0:
            raise ValueError("v0 must be positive")


def simulate_heston_spot_paths(
    *,
    spot: float,
    rate: float,
    maturity: float,
    steps: int,
    paths: int,
    params: HestonParams,
    antithetic: bool = True,
    moment_matching: bool = False,
    seed: int | None = None,
) -> tuple[FloatArray, FloatArray]:
    """
    Simulate spot and variance paths under the Heston model.
    """

    grid = build_time_grid(maturity=float(maturity), steps=int(steps))
    model = HestonModel(
        kappa=float(params.kappa),
        theta=float(params.theta),
        vol_of_vol=float(params.vol_of_vol),
        rho=float(params.rho),
        drift=float(rate),
        seed=seed,
        antithetic=bool(antithetic),
        moment_matching=bool(moment_matching),
    )
    return model.simulate_paths(
        spot=float(spot),
        variance=float(params.v0),
        time_grid=grid,
        paths=int(paths),
    )


def mc_call_prices(
    spot_paths: FloatArray,
    strikes: FloatArray,
    *,
    discount_factor: float,
    forward: float | None = None,
) -> FloatArray:
    """
    Monte Carlo prices for European calls from spot paths.
    """

    if spot_paths.ndim != 2:
        raise ValueError("spot_paths must be two-dimensional")
    if strikes.ndim != 1:
        raise ValueError("strikes must be one-dimensional")

    df = float(discount_factor)
    if df <= 0:
        raise ValueError("discount_factor must be positive")

    terminal = spot_paths[:, -1].astype(float, copy=False)
    payoffs = np.maximum(terminal[:, None] - strikes[None, :], 0.0)

    mean_payoffs = np.mean(payoffs, axis=0)
    if forward is None:
        prices = df * mean_payoffs
        return prices.astype(float, copy=False)

    fwd = float(forward)
    y = terminal
    y_mean = float(np.mean(y))
    y_center = y - y_mean
    var_y = float(np.mean(y_center**2))
    if var_y <= 0.0:
        prices = df * mean_payoffs
        return prices.astype(float, copy=False)

    x_center = payoffs - mean_payoffs[None, :]
    cov_xy = np.mean(x_center * y_center[:, None], axis=0)
    beta = cov_xy / var_y
    adj_means = mean_payoffs + beta * (fwd - y_mean)
    prices = df * adj_means
    return prices.astype(float, copy=False)

## `code/dxlib/calibration.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 31 - Market-Based Valuation.

Simple Monte Carlo based calibration routines (no SciPy dependency).

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

import numpy as np
import pandas as pd

from .black_scholes import implied_vol_forward
from .heston import HestonParams, mc_call_prices, simulate_heston_spot_paths
from .processes import JumpDiffusion, build_time_grid

__all__ = [
    "CalibrationBounds",
    "CalibrationDiagnostics",
    "JumpDiffusionBounds",
    "JumpDiffusionParams",
    "evaluate_jump_diffusion_fit_table",
    "calibrate_jump_diffusion_single_expiry",
    "evaluate_heston_fit_table",
    "calibrate_heston_global",
    "calibrate_heston_local_v0",
    "calibrate_heston_local_theta_v0_rho",
    "calibrate_heston_local_v0_rho",
    "spot_from_forwards",
]


@dataclass(frozen=True, slots=True)
class CalibrationBounds:
    kappa: tuple[float, float] = (0.5, 8.0)
    theta: tuple[float, float] = (0.005, 0.20)
    vol_of_vol: tuple[float, float] = (0.05, 1.25)
    rho: tuple[float, float] = (-0.95, -0.05)
    v0: tuple[float, float] = (0.005, 0.20)


@dataclass(frozen=True, slots=True)
class CalibrationDiagnostics:
    seed: int
    n_candidates: int
    n_refine: int
    evaluations: int
    failures: int
    best_loss: float
    best_stage: str
    loss_path: tuple[float, ...]


@dataclass(frozen=True, slots=True)
class JumpDiffusionParams:
    volatility: float
    jump_intensity: float
    jump_mean: float
    jump_std: float

    def __post_init__(self) -> None:
        if self.volatility < 0:
            raise ValueError("volatility must be non-negative")
        if self.jump_intensity < 0:
            raise ValueError("jump_intensity must be non-negative")
        if self.jump_std < 0:
            raise ValueError("jump_std must be non-negative")


@dataclass(frozen=True, slots=True)
class JumpDiffusionBounds:
    volatility: tuple[float, float] = (0.05, 0.60)
    jump_intensity: tuple[float, float] = (0.0, 8.0)
    # Widened from (-0.25, 0.05).
    jump_mean: tuple[float, float] = (-0.40, 0.10)
    jump_std: tuple[float, float] = (0.01, 0.80)  # widened from (0.05, 0.60)


MIN_CALIBRATION_STEPS = 24


def _require_surface_columns(surface_df: pd.DataFrame) -> None:
    required = {"EXPIR_DATE", "TTM", "FWD", "DF", "STRIKE", "IMPL_VOL", "MNY"}
    missing = required - set(surface_df.columns)
    if missing:
        msg = f"surface_df missing required columns: {sorted(missing)}"
        raise ValueError(msg)


def spot_from_forwards(surface_df: pd.DataFrame) -> float:
    """
    Estimate a single spot from per-expiry forward levels under q=0.
    """

    _require_surface_columns(surface_df)
    spot = surface_df["FWD"].to_numpy(dtype=float) * surface_df["DF"].to_numpy(
        dtype=float
    )
    return float(np.median(spot))


def _calc_model_fit_table(
    surface_df: pd.DataFrame,
    *,
    rate: float,
    params: HestonParams | JumpDiffusionParams,
    paths: int,
    steps_per_year: int,
    seed: int,
) -> pd.DataFrame:
    """
    Unified evaluation logic for Heston and Jump Diffusion models.
    """

    _require_surface_columns(surface_df)
    tables: list[pd.DataFrame] = []
    implied: list[float] = []

    for _, sub in surface_df.groupby("EXPIR_DATE", sort=True):
        ttm = float(sub["TTM"].iloc[0])
        spot0 = float(sub["FWD"].iloc[0]) * float(sub["DF"].iloc[0])
        steps = _steps_for_ttm(ttm, steps_per_year)
        iter_seed = seed + int(round(ttm * 10_000))

        if isinstance(params, HestonParams):
            spot_paths, _ = simulate_heston_spot_paths(
                spot=spot0,
                rate=rate,
                maturity=ttm,
                steps=steps,
                paths=paths,
                params=params,
                seed=iter_seed,
                antithetic=True,
                moment_matching=False,
            )
        else:
            grid = build_time_grid(maturity=ttm, steps=steps)
            model = JumpDiffusion(
                drift=rate,
                volatility=float(params.volatility),
                jump_intensity=float(params.jump_intensity),
                jump_mean=float(params.jump_mean),
                jump_std=float(params.jump_std),
                seed=iter_seed,
                antithetic=True,
                moment_matching=False,
            )
            spot_paths = model.simulate_paths(
                spot=spot0,
                time_grid=grid,
                paths=int(paths),
            )

        work = sub.sort_values("STRIKE").copy()
        strikes = work["STRIKE"].to_numpy(dtype=float)
        fwd = float(sub["FWD"].iloc[0])
        df = float(sub["DF"].iloc[0])

        prices = mc_call_prices(
            spot_paths,
            strikes,
            discount_factor=df,
            forward=fwd,
        )

        work["MODEL_PRICE_RAW"] = prices
        intrinsic = df * np.maximum(fwd - strikes, 0.0)
        work["INTRINSIC"] = intrinsic
        work["FLOORED_TO_INTRINSIC"] = prices < intrinsic
        work["MODEL_PRICE"] = np.maximum(prices, intrinsic)
        work["TV"] = work["MODEL_PRICE"] - work["INTRINSIC"]

        for _, row in work.iterrows():
            iv = implied_vol_forward(
                price=float(row["MODEL_PRICE"]),
                forward=float(row["FWD"]),
                strike=float(row["STRIKE"]),
                maturity=float(row["TTM"]),
                option_type="call"
                if str(row.get("CALIBRATE_TO", "CALL")).upper() == "CALL"
                else "put",
                discount_factor=float(row["DF"]),
            )
            implied.append(iv)
        tables.append(work)

    out = pd.concat(tables, axis=0, ignore_index=True)
    out["MODEL_IV"] = np.array(implied, dtype=float)
    out["IV_ERR"] = out["MODEL_IV"] - out["IMPL_VOL"]
    out["IV_ERR2"] = out["IV_ERR"] ** 2
    return out


def _clip(x: float, low: float, high: float) -> float:
    return float(min(max(float(x), float(low)), float(high)))


def _steps_for_ttm(ttm: float, steps_per_year: int) -> int:
    steps = int(round(float(steps_per_year) * float(ttm)))
    return max(steps, MIN_CALIBRATION_STEPS)


def _with_diagnostics(
    table: pd.DataFrame,
    diagnostics: CalibrationDiagnostics | dict[object, CalibrationDiagnostics],
) -> pd.DataFrame:
    out = table.copy()
    out.attrs["diagnostics"] = diagnostics
    return out


def evaluate_heston_fit_table(
    surface_df: pd.DataFrame,
    *,
    rate: float,
    params: HestonParams,
    paths: int,
    steps_per_year: int,
    seed: int,
) -> pd.DataFrame:
    """
    Evaluate model prices and implied vols for a forward-based small surface.
    """

    return _calc_model_fit_table(
        surface_df,
        rate=rate,
        params=params,
        paths=paths,
        steps_per_year=steps_per_year,
        seed=seed,
    )


def evaluate_jump_diffusion_fit_table(
    surface_df: pd.DataFrame,
    *,
    rate: float,
    params: JumpDiffusionParams,
    paths: int,
    steps_per_year: int,
    seed: int,
) -> pd.DataFrame:
    """
    Evaluate model prices and implied vols under a jump diffusion model.
    """

    return _calc_model_fit_table(
        surface_df,
        rate=rate,
        params=params,
        paths=paths,
        steps_per_year=steps_per_year,
        seed=seed,
    )


def calibrate_jump_diffusion_single_expiry(
    surface_df: pd.DataFrame,
    *,
    rate: float,
    bounds: JumpDiffusionBounds | None = None,
    paths: int = 25_000,
    steps_per_year: int = 240,
    n_candidates: int = 80,
    n_refine: int = 40,
    seed: int = 29,
) -> tuple[JumpDiffusionParams, pd.DataFrame]:
    """
    Calibrate a jump diffusion model to one expiry surface slice.
    """

    _require_surface_columns(surface_df)
    expiries = surface_df["EXPIR_DATE"].drop_duplicates().to_list()
    if len(expiries) != 1:
        raise ValueError("surface_df must contain exactly one expiry")

    if bounds is None:
        bounds = JumpDiffusionBounds()

    rng = np.random.default_rng(seed)

    def sample(low: float, high: float, n: int) -> np.ndarray:
        return low + (high - low) * rng.random(n)

    candidates = pd.DataFrame(
        {
            "volatility": sample(*bounds.volatility, n_candidates),
            "jump_intensity": sample(*bounds.jump_intensity, n_candidates),
            "jump_mean": sample(*bounds.jump_mean, n_candidates),
            "jump_std": sample(*bounds.jump_std, n_candidates),
        }
    )

    best_params: JumpDiffusionParams | None = None
    best_loss = float("inf")
    best_table: pd.DataFrame | None = None
    evaluations = 0
    failures = 0
    best_stage = "global"
    loss_path: list[float] = []

    for _, row in candidates.iterrows():
        params = JumpDiffusionParams(
            volatility=float(row["volatility"]),
            jump_intensity=float(row["jump_intensity"]),
            jump_mean=float(row["jump_mean"]),
            jump_std=float(row["jump_std"]),
        )
        try:
            table = _calc_model_fit_table(
                surface_df,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            loss = float(table["IV_ERR2"].mean())
        except Exception:
            failures += 1
            continue
        evaluations += 1
        loss_path.append(loss)
        if loss < best_loss:
            best_loss = loss
            best_params = params
            best_table = table

    if best_params is None or best_table is None:
        raise RuntimeError(
            "JD calibration failed: no candidate produced a result"
        )

    for step in range(int(n_refine)):
        scale = 0.35 * (0.98**step)
        vol = best_params.volatility * float(
            rng.lognormal(mean=0.0, sigma=scale)
        )
        lam = best_params.jump_intensity * float(
            rng.lognormal(mean=0.0, sigma=scale)
        )
        m_jump = best_params.jump_mean + float(rng.normal(0.0, 0.08 * scale))
        s_jump = best_params.jump_std * float(
            rng.lognormal(mean=0.0, sigma=scale)
        )
        params = JumpDiffusionParams(
            volatility=_clip(vol, *bounds.volatility),
            jump_intensity=_clip(lam, *bounds.jump_intensity),
            jump_mean=_clip(m_jump, *bounds.jump_mean),
            jump_std=_clip(s_jump, *bounds.jump_std),
        )
        try:
            table = _calc_model_fit_table(
                surface_df,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            loss = float(table["IV_ERR2"].mean())
        except Exception:
            failures += 1
            continue
        evaluations += 1
        loss_path.append(loss)
        if loss < best_loss:
            best_loss = loss
            best_params = params
            best_table = table
            best_stage = "refine"

    diagnostics = CalibrationDiagnostics(
        seed=int(seed),
        n_candidates=int(n_candidates),
        n_refine=int(n_refine),
        evaluations=evaluations,
        failures=failures,
        best_loss=best_loss,
        best_stage=best_stage,
        loss_path=tuple(loss_path),
    )
    return best_params, _with_diagnostics(best_table, diagnostics)


def calibrate_heston_global(
    surface_df: pd.DataFrame,
    *,
    rate: float,
    bounds: CalibrationBounds | None = None,
    paths: int = 25_000,
    steps_per_year: int = 200,
    n_candidates: int = 80,
    n_refine: int = 40,
    seed: int = 17,
) -> tuple[HestonParams, pd.DataFrame]:
    """
    Global calibration to the full 3x5 surface via random search.
    """

    if bounds is None:
        bounds = CalibrationBounds()

    rng = np.random.default_rng(seed)

    def sample(low: float, high: float, n: int) -> np.ndarray:
        return low + (high - low) * rng.random(n)

    candidates = pd.DataFrame(
        {
            "kappa": sample(*bounds.kappa, n_candidates),
            "theta": sample(*bounds.theta, n_candidates),
            "vol_of_vol": sample(*bounds.vol_of_vol, n_candidates),
            "rho": sample(*bounds.rho, n_candidates),
            "v0": sample(*bounds.v0, n_candidates),
        }
    )

    best_params: HestonParams | None = None
    best_loss = float("inf")
    best_table: pd.DataFrame | None = None
    evaluations = 0
    failures = 0
    best_stage = "global"
    loss_path: list[float] = []

    for _, row in candidates.iterrows():
        params = HestonParams(
            kappa=float(row["kappa"]),
            theta=float(row["theta"]),
            vol_of_vol=float(row["vol_of_vol"]),
            rho=float(row["rho"]),
            v0=float(row["v0"]),
        )
        try:
            table = _calc_model_fit_table(
                surface_df,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            loss = float(table["IV_ERR2"].mean())
        except Exception:
            failures += 1
            continue
        evaluations += 1
        loss_path.append(loss)

        if loss < best_loss:
            best_loss = loss
            best_params = params
            best_table = table

    if best_params is None or best_table is None:
        msg = "Calibration failed: no candidate produced a result"
        raise RuntimeError(msg)

    for step in range(int(n_refine)):
        scale = 0.35 * (0.98**step)
        kappa = best_params.kappa * float(rng.lognormal(mean=0.0, sigma=scale))
        theta = best_params.theta * float(rng.lognormal(mean=0.0, sigma=scale))
        vol_of_vol = best_params.vol_of_vol * float(
            rng.lognormal(mean=0.0, sigma=scale)
        )
        v0 = best_params.v0 * float(rng.lognormal(mean=0.0, sigma=scale))
        rho = best_params.rho + float(rng.normal(0.0, 0.15 * scale))

        params = HestonParams(
            kappa=_clip(kappa, *bounds.kappa),
            theta=_clip(theta, *bounds.theta),
            vol_of_vol=_clip(vol_of_vol, *bounds.vol_of_vol),
            rho=_clip(rho, *bounds.rho),
            v0=_clip(v0, *bounds.v0),
        )
        try:
            table = _calc_model_fit_table(
                surface_df,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            loss = float(table["IV_ERR2"].mean())
        except Exception:
            failures += 1
            continue
        evaluations += 1
        loss_path.append(loss)

        if loss < best_loss:
            best_loss = loss
            best_params = params
            best_table = table
            best_stage = "refine"

    diagnostics = CalibrationDiagnostics(
        seed=int(seed),
        n_candidates=int(n_candidates),
        n_refine=int(n_refine),
        evaluations=evaluations,
        failures=failures,
        best_loss=best_loss,
        best_stage=best_stage,
        loss_path=tuple(loss_path),
    )
    return best_params, _with_diagnostics(best_table, diagnostics)


def calibrate_heston_local_v0(
    surface_df: pd.DataFrame,
    *,
    global_params: HestonParams,
    rate: float,
    bounds: CalibrationBounds | None = None,
    paths: int = 25_000,
    steps_per_year: int = 200,
    seed: int = 19,
    v0_grid: np.ndarray | None = None,
    mode: str = "atm",
) -> tuple[dict[object, float], pd.DataFrame]:
    """
    Re-fit v0 per expiry while keeping other parameters fixed.
    """

    mode = str(mode).strip().lower()
    if mode not in {"atm", "smile"}:
        raise ValueError("mode must be 'atm' or 'smile'")

    if bounds is None:
        bounds = CalibrationBounds()

    if v0_grid is None:
        base = float(global_params.v0)
        v0_grid = np.array(
            [0.25 * base, 0.5 * base, base, 2.0 * base, 4.0 * base]
        )
        v0_grid = np.clip(v0_grid, *bounds.v0)

    tables: list[pd.DataFrame] = []
    v0_map: dict[object, float] = {}
    diagnostics_map: dict[object, CalibrationDiagnostics] = {}

    for expiry, sub in surface_df.groupby("EXPIR_DATE", sort=True):
        best_v0 = float(v0_grid[0])
        best_loss = float("inf")
        best_table: pd.DataFrame | None = None
        evaluations = 0
        failures = 0
        loss_path: list[float] = []

        target = sub.copy()
        if mode == "atm":
            target = target.assign(abs_mny=(target["MNY"] - 1.0).abs())
            target = target.sort_values("abs_mny").head(1)

        for v0 in v0_grid:
            params = HestonParams(
                kappa=global_params.kappa,
                theta=global_params.theta,
                vol_of_vol=global_params.vol_of_vol,
                rho=global_params.rho,
                v0=float(v0),
            )
            try:
                table = _calc_model_fit_table(
                    target,
                    rate=rate,
                    params=params,
                    paths=paths,
                    steps_per_year=steps_per_year,
                    seed=seed,
                )
                loss = float(table["IV_ERR2"].mean())
            except Exception:
                failures += 1
                continue
            evaluations += 1
            loss_path.append(loss)

            if loss < best_loss:
                best_loss = loss
                best_v0 = float(v0)
                best_table = table

        if best_table is None:
            raise RuntimeError(f"Local calibration failed for expiry {expiry}")

        v0_map[expiry] = best_v0
        full_params = HestonParams(
            kappa=global_params.kappa,
            theta=global_params.theta,
            vol_of_vol=global_params.vol_of_vol,
            rho=global_params.rho,
            v0=best_v0,
        )
        full_table = _calc_model_fit_table(
            sub,
            rate=rate,
            params=full_params,
            paths=paths,
            steps_per_year=steps_per_year,
            seed=seed,
        )
        tables.append(full_table)
        diagnostics_map[expiry] = CalibrationDiagnostics(
            seed=int(seed),
            n_candidates=int(len(v0_grid)),
            n_refine=0,
            evaluations=evaluations,
            failures=failures,
            best_loss=best_loss,
            best_stage=mode,
            loss_path=tuple(loss_path),
        )

    out = pd.concat(tables, axis=0, ignore_index=True)
    return v0_map, _with_diagnostics(out, diagnostics_map)


def calibrate_heston_local_v0_rho(
    surface_df: pd.DataFrame,
    *,
    global_params: HestonParams,
    rate: float,
    bounds: CalibrationBounds | None = None,
    paths: int = 25_000,
    steps_per_year: int = 200,
    seed: int = 23,
    v0_grid: np.ndarray | None = None,
    rho_grid: np.ndarray | None = None,
    n_candidates: int = 40,
    n_refine: int = 25,
) -> tuple[dict[object, tuple[float, float]], pd.DataFrame]:
    """
    Local refinement: re-fit v0 and rho per expiry (smile fit).
    """

    if bounds is None:
        bounds = CalibrationBounds()

    param_map: dict[object, tuple[float, float]] = {}
    tables: list[pd.DataFrame] = []
    diagnostics_map: dict[object, CalibrationDiagnostics] = {}

    rng = np.random.default_rng(seed)

    for expiry, sub in surface_df.groupby("EXPIR_DATE", sort=True):
        best_v0 = float(global_params.v0)
        best_rho = float(global_params.rho)
        best_loss = float("inf")
        evaluations = 0
        failures = 0
        if v0_grid is not None and rho_grid is not None:
            best_stage = "grid"
        else:
            best_stage = "global"
        loss_path: list[float] = []

        def loss_for(v0: float, rho: float) -> float:
            params = HestonParams(
                kappa=global_params.kappa,
                theta=global_params.theta,
                vol_of_vol=global_params.vol_of_vol,
                rho=float(rho),
                v0=float(v0),
            )
            table = _calc_model_fit_table(
                sub,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            return float(table["IV_ERR2"].mean())

        if v0_grid is not None and rho_grid is not None:
            for v0 in v0_grid:
                for rho in rho_grid:
                    try:
                        loss = loss_for(float(v0), float(rho))
                    except Exception:
                        failures += 1
                        continue
                    evaluations += 1
                    loss_path.append(loss)
                    if loss < best_loss:
                        best_loss = loss
                        best_v0 = float(v0)
                        best_rho = float(rho)
        else:
            base_v0 = float(global_params.v0)
            base_rho = float(global_params.rho)
            for _ in range(int(n_candidates)):
                v0 = base_v0 * float(rng.lognormal(mean=0.0, sigma=0.50))
                v0 = _clip(v0, *bounds.v0)
                rho = base_rho + float(rng.normal(0.0, 0.25))
                rho = _clip(rho, *bounds.rho)
                try:
                    loss = loss_for(v0, rho)
                except Exception:
                    failures += 1
                    continue
                evaluations += 1
                loss_path.append(loss)
                if loss < best_loss:
                    best_loss = loss
                    best_v0 = v0
                    best_rho = rho

            for step in range(int(n_refine)):
                scale = 0.35 * (0.98**step)
                v0 = best_v0 * float(rng.lognormal(mean=0.0, sigma=scale))
                v0 = _clip(v0, *bounds.v0)
                rho = best_rho + float(rng.normal(0.0, 0.15 * scale))
                rho = _clip(rho, *bounds.rho)
                try:
                    loss = loss_for(v0, rho)
                except Exception:
                    failures += 1
                    continue
                evaluations += 1
                loss_path.append(loss)
                if loss < best_loss:
                    best_loss = loss
                    best_v0 = v0
                    best_rho = rho
                    best_stage = "refine"

        if evaluations == 0:
            raise RuntimeError(f"Local calibration failed for expiry {expiry}")

        param_map[expiry] = (best_v0, best_rho)
        best_params = HestonParams(
            kappa=global_params.kappa,
            theta=global_params.theta,
            vol_of_vol=global_params.vol_of_vol,
            rho=best_rho,
            v0=best_v0,
        )
        table = _calc_model_fit_table(
            sub,
            rate=rate,
            params=best_params,
            paths=paths,
            steps_per_year=steps_per_year,
            seed=seed,
        )
        tables.append(table)
        diagnostics_map[expiry] = CalibrationDiagnostics(
            seed=int(seed),
            n_candidates=int(n_candidates)
            if v0_grid is None or rho_grid is None
            else int(len(v0_grid) * len(rho_grid)),
            n_refine=int(n_refine),
            evaluations=evaluations,
            failures=failures,
            best_loss=best_loss,
            best_stage=best_stage,
            loss_path=tuple(loss_path),
        )

    out = pd.concat(tables, axis=0, ignore_index=True)
    return param_map, _with_diagnostics(out, diagnostics_map)


def calibrate_heston_local_theta_v0_rho(
    surface_df: pd.DataFrame,
    *,
    global_params: HestonParams,
    rate: float,
    bounds: CalibrationBounds | None = None,
    paths: int = 25_000,
    steps_per_year: int = 200,
    seed: int = 23,
    n_candidates: int = 40,
    n_refine: int = 25,
) -> tuple[dict[object, tuple[float, float, float]], pd.DataFrame]:
    """
    Local refinement: re-fit theta, v0, and rho per expiry.
    """

    if bounds is None:
        bounds = CalibrationBounds()

    param_map: dict[object, tuple[float, float, float]] = {}
    tables: list[pd.DataFrame] = []
    diagnostics_map: dict[object, CalibrationDiagnostics] = {}

    rng = np.random.default_rng(seed)

    for expiry, sub in surface_df.groupby("EXPIR_DATE", sort=True):
        best_theta = float(global_params.theta)
        best_v0 = float(global_params.v0)
        best_rho = float(global_params.rho)
        best_loss = float("inf")
        evaluations = 0
        failures = 0
        best_stage = "global"
        loss_path: list[float] = []

        def loss_for(theta: float, v0: float, rho: float) -> float:
            params = HestonParams(
                kappa=global_params.kappa,
                theta=float(theta),
                vol_of_vol=global_params.vol_of_vol,
                rho=float(rho),
                v0=float(v0),
            )
            table = _calc_model_fit_table(
                sub,
                rate=rate,
                params=params,
                paths=paths,
                steps_per_year=steps_per_year,
                seed=seed,
            )
            return float(table["IV_ERR2"].mean())

        for _ in range(int(n_candidates)):
            theta = global_params.theta * float(
                rng.lognormal(mean=0.0, sigma=0.45)
            )
            theta = _clip(theta, *bounds.theta)
            v0 = global_params.v0 * float(rng.lognormal(mean=0.0, sigma=0.50))
            v0 = _clip(v0, *bounds.v0)
            rho = global_params.rho + float(rng.normal(0.0, 0.25))
            rho = _clip(rho, *bounds.rho)
            try:
                loss = loss_for(theta, v0, rho)
            except Exception:
                failures += 1
                continue
            evaluations += 1
            loss_path.append(loss)
            if loss < best_loss:
                best_loss = loss
                best_theta = theta
                best_v0 = v0
                best_rho = rho

        for step in range(int(n_refine)):
            scale = 0.35 * (0.98**step)
            theta = best_theta * float(rng.lognormal(mean=0.0, sigma=scale))
            theta = _clip(theta, *bounds.theta)
            v0 = best_v0 * float(rng.lognormal(mean=0.0, sigma=scale))
            v0 = _clip(v0, *bounds.v0)
            rho = best_rho + float(rng.normal(0.0, 0.15 * scale))
            rho = _clip(rho, *bounds.rho)
            try:
                loss = loss_for(theta, v0, rho)
            except Exception:
                failures += 1
                continue
            evaluations += 1
            loss_path.append(loss)
            if loss < best_loss:
                best_loss = loss
                best_theta = theta
                best_v0 = v0
                best_rho = rho
                best_stage = "refine"

        if evaluations == 0:
            raise RuntimeError(f"Local calibration failed for expiry {expiry}")

        param_map[expiry] = (best_theta, best_v0, best_rho)
        best_params = HestonParams(
            kappa=global_params.kappa,
            theta=best_theta,
            vol_of_vol=global_params.vol_of_vol,
            rho=best_rho,
            v0=best_v0,
        )
        table = _calc_model_fit_table(
            sub,
            rate=rate,
            params=best_params,
            paths=paths,
            steps_per_year=steps_per_year,
            seed=seed,
        )
        tables.append(table)
        diagnostics_map[expiry] = CalibrationDiagnostics(
            seed=int(seed),
            n_candidates=int(n_candidates),
            n_refine=int(n_refine),
            evaluations=evaluations,
            failures=failures,
            best_loss=best_loss,
            best_stage=best_stage,
            loss_path=tuple(loss_path),
        )

    out = pd.concat(tables, axis=0, ignore_index=True)
    return param_map, _with_diagnostics(out, diagnostics_map)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
